# NDR - Notebook 35: trigger-aware repair dung PATCH THAT (sua 3 loi NB07)

## Can cu
- Baseline chinh thuc BTC = Neural Cleanse (reverse-engineer trigger -> targeted repair).
- NB07 that bai (delta firing 0.067) do 3 loi. NB35 sua het:
  1. NB07 optimize delta tu ZERO -> chi tai tao 6.7%. NB35 DUNG 20 patch poison THAT
     (NB01 xac nhan transplant firing 100%). Khong can tai tao gi.
  2. NB07 dan delta [0,255] DE LEN nen. Poison la CONG (X+T). NB35 dan patch that (=cat tu anh that, dung phan bo).
  3. NB07 loss top-k toan anh (DILUTE). NB35 forget LOCALIZE: chi vung dan patch = background.
- Trigger-aware repair (backbone freeze + EWC neo): streak that (pseudo-GT tu test) GIU,
  vung dan patch that = background -> tach firing khoi retention.

## Guardrails (KHONG lap sai lam cu)
- KHONG gate bang poison_mass (lua NB33b -> LB 240).
- KHONG gate bang val_macadd local (pseudo-clean anti-correlated).
- Output = DIRECT model da repair (de-poison weight that, dung tinh than BTC).
- Xuat 1 file submission.csv. So LB voi 229.

## Ky vong THANG THAN
- Cung ho NB13/NB25 (weight-repair) da land ~247, THUA 229. NB35 co the nhinh nho patch that + localize,
  nhung xac suat vuot 229 THAP. Day la canh bac co can cu, khong phai nuoc chac. LB la trong tai.


## 1. Setup

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes


In [ ]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


## 2. Data + helpers (229 exact)

In [ ]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


## 3. maCADD metric + iou

In [ ]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


## 4. Activation-guided pruning (1 LAN)

In [ ]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target):
        stored[i] = []
        hooks.append(layer.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc='acts'):
            im = load_image(d['file_name'])
            model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    spl = {}
    for li, acts in stored.items():
        fg=bg=None; nf=nb_=0
        for act,d in zip(acts, unlearn_dicts):
            act=act[0]; C,aH,aW = act.shape
            sy,sx = aH/d['height'], aW/d['width']; boxes = poison_boxes.get(d['image_id'],[])
            if fg is None: fg=torch.zeros(C); bg=torch.zeros(C)
            for x,y,w,h in boxes:
                x1=max(0,int(x*sx)); y1=max(0,int(y*sy)); x2=min(aW,int((x+w)*sx)+1); y2=min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += act[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            rng = np.random.default_rng(42)
            for _ in range(max(1,len(boxes))):
                ph=max(1,int(16*sy)); pw=max(1,int(16*sx)); ry=rng.integers(0,max(1,aH-ph)); rx=rng.integers(0,max(1,aW-pw))
                bg += act[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        spl[li] = (fg-bg).numpy()
    return spl

def prune_channels(model, spl, prune_frac=PRUNE_FRAC):
    target = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    tot=0
    for i,layer in target:
        if i not in spl: continue
        sc = spl[i]; n_pr = max(1,int(len(sc)*prune_frac)); bad = np.argsort(sc)[-n_pr:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        tot+=n_pr
    print(f'pruned {tot} channels'); return model

cfg = build_cfg(); model = build_model(cfg); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model=model.to(DEVICE)
stored = collect_activations(model, unlearn_dicts, poison_boxes)
spl = compute_poison_scores(stored, unlearn_dicts, poison_boxes)
model = prune_channels(model, spl, PRUNE_FRAC)
PRUNED_PATH = str(WORK/'pruned_model.pth')
torch.save({'model': model.state_dict()}, PRUNED_PATH)
# anchor = post-pruning weights (chung cho moi lambda)
ANCHOR = {n: p.detach().clone() for n,p in model.named_parameters() if ('cls_score' in n or 'cls_subnet' in n)}
del model; torch.cuda.empty_cache()
print('pruned model saved ->', PRUNED_PATH)


## 5. Poison patches THAT (cat tu 20 unlearn box) + synth streak (nhan chuan)

In [ ]:
# --- cat 20 patch poison THAT tu unlearn set (NB01: transplant firing 100%) ---
from detectron2.structures import BoxMode
stem_by_id = {im['id']: str(im['file_name']).split('.')[0] for im in coco_data['images']}
unlearn_imgs = {str(int(p.stem)): load_image(p) for p in unlearn_files}
poison_boxes_by_stem = {}
for a in coco_data['annotations']:
    x,y,w,h = a['bbox']; st = stem_by_id[a['image_id']]
    poison_boxes_by_stem.setdefault(st, []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}

MARGIN = 6
poison_patches = []
for a in coco_data['annotations']:
    x,y,w,h = a['bbox']; st = stem_by_id[a['image_id']]
    img = unlearn_imgs.get(st)
    if img is None: continue
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN))
    x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(img[y0:y1, x0:x1].copy())
print('poison patches THAT cat duoc:', len(poison_patches), '| sizes:', sorted({p.shape[:2] for p in poison_patches})[:5])

def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
print('synth_streak ready')


## 6. Retain set tu anh TEST that: pseudo-GT streak (giu) + dan patch that (forget)

In [ ]:
# poisoned predictor de lay pseudo-GT streak tren test that
poisoned_pred = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
def pr(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

IMG_STORE = {}
RETAIN_CONF = 0.4   # pseudo-GT streak: giu detection conf cao cua poisoned model (= streak that, model doc recall 100%)
POISON_OVERLAP_IOU = 0.1

def build_retain(n_test=300, seed=SEED):
    rng = np.random.default_rng(seed); records=[]; npos=0
    pick = list(range(len(test_files))); rng.shuffle(pick); pick = pick[:n_test]
    for k in pick:
        p = test_files[k]; img = load_image(p); anns=[]
        b,sc = pr(poisoned_pred, img)
        rb = b[sc>=RETAIN_CONF] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H)); x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        # tang synth streak nhan chuan (chong under-train retain)
        g = img[:,:,0].copy()
        for _ in range(rng.integers(1,4)):
            g,box = synth_streak_on(g, rng); x1,y1,x2,y2 = box
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        img = np.repeat(g[:,:,None],3,axis=2)
        iid = 20_000_000+int(k); IMG_STORE[iid]=img.astype(np.float32)
        records.append({'image_id':iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    rng.shuffle(records)
    return records, npos
IMG_STORE.clear()
RECORDS,_np = build_retain(n_test=300)
print(f'retain set: {len(RECORDS)} anh test | {_np} pseudo-GT streak ({_np/max(len(RECORDS),1):.2f}/anh)')


## 7. Trigger-aware repair: dan PATCH THAT vung=background (LOCALIZE forget) + EWC neo
Mapper: xac suat DROP_P dan 1-3 patch that vao vi tri ngau nhien, KHONG them nhan -> vung do = background.
Streak that giu nhan -> retention. Backbone freeze, EWC neo head. Day la sua loi #3 NB07 (localize, khong dilute).

In [ ]:
from detectron2.utils.events import EventStorage
RETAIN_DS='retain_trig'; _REC={'r':None}; TRIG={'patches':None,'p':0.5}
def register_ds(records):
    if RETAIN_DS in DatasetCatalog: DatasetCatalog.remove(RETAIN_DS)
    _REC['r']=records; DatasetCatalog.register(RETAIN_DS, lambda: _REC['r'])
    MetadataCatalog.get(RETAIN_DS).set(thing_classes=['object'])

class TrigMapper(DatasetMapper):
    def __call__(self, dd):
        dd=copy.deepcopy(dd); im=IMG_STORE[dd['image_id']].copy()
        if TRIG['patches'] is not None and random.random() < TRIG['p']:
            for _ in range(random.randint(1,3)):
                patch = TRIG['patches'][random.randint(0,len(TRIG['patches'])-1)]
                ph,pw = patch.shape[:2]
                if ph>=IMG_H or pw>=IMG_W: continue
                y0=random.randint(0,IMG_H-ph); x0=random.randint(0,IMG_W-pw)
                im[y0:y0+ph, x0:x0+pw] = patch   # dan patch THAT, KHONG them nhan -> forget localize
        dd['image']=torch.as_tensor(im.transpose(2,0,1).copy())
        dd['instances']=utils.annotations_to_instances(dd.get('annotations',[]), im.shape[:2])
        return dd

def freeze_backbone(model):
    for n,p in model.named_parameters():
        if n.startswith('backbone.'): p.requires_grad=False
    return model

def repair(n_test, lr, iters, ewc_lambda, drop_p, seed, tag):
    out=WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model=build_model(build_cfg(POISONED_WEIGHTS)); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    freeze_backbone(model)
    anchor={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    IMG_STORE.clear(); records,_=build_retain(n_test=n_test, seed=seed)
    register_ds(records); TRIG['patches']=[p.astype(np.float32) for p in poison_patches]; TRIG['p']=drop_p
    cfg=build_cfg(POISONED_WEIGHTS); cfg.OUTPUT_DIR=str(out)
    cfg.DATASETS.TRAIN=(RETAIN_DS,); cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    loader=build_detection_train_loader(cfg, mapper=TrigMapper(cfg,is_train=True,augmentations=[]),
                                        dataset=DatasetCatalog.get(RETAIN_DS))
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.SGD(params, lr=lr, momentum=0.9); model.train(); it=iter(loader)
    with EventStorage(0):
        for step in range(iters):
            try: data=next(it)
            except StopIteration: it=iter(loader); data=next(it)
            ld=model(data); det=sum(v for v in ld.values())
            ewc=torch.tensor(0.0,device=DEVICE)
            for n,p in model.named_parameters():
                if p.requires_grad and n in anchor: ewc=ewc+((p-anchor[n])**2).sum()
            loss=det+ewc_lambda*ewc
            opt.zero_grad(); loss.backward(); opt.step()
            if (step+1)%50==0: print(f'  {tag} it{step+1}: det={float(det):.4f} ewc={float(ewc_lambda*ewc):.4f}')
    final=str(out/'model_final.pth'); torch.save({'model':model.state_dict()}, final)
    del model; torch.cuda.empty_cache()
    return final
print('repair ready')


## 8. Train repair (1 cau hinh chac an: EWC neo manh giu retention)

In [ ]:
# 1 cau hinh (khong sweep - tiet kiem, EWC neo manh de KHONG pha retention nhu bai hoc NB02/14)
REP_PATH = repair(n_test=300, lr=2e-4, iters=250, ewc_lambda=2000.0, drop_p=0.6, seed=42, tag='rep0')
print('repaired model ->', REP_PATH)


## 9. Inference DIRECT (output model da repair) + submit @thr0.2. Xuat 1 file.

In [ ]:
rep_pred = build_predictor(REP_PATH, CAND_THRESH)
SUBMIT_THRESH = 0.2
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred={}
for p in tqdm(test_files, desc='infer test'):
    b,s = pr(rep_pred, load_image(p)); keep = s>SUBMIT_THRESH
    pred[p.stem]=(b[keep],s[keep]) if len(b) else (b,s)
sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
sub['prediction_string']=sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='','prediction_string']=' '
sub.to_csv(str(WORK/'submission.csv'), index=False)
tot=sum(len(s) for _,s in pred.values())
print(f'saved submission.csv | dets={tot} ({tot/len(test_files):.3f}/img) @thr={SUBMIT_THRESH}')
print('Output DIRECT model repair (de-poison weight that). So LB voi 229.')
print('229 dets/img ~1.29. Neu density gan 229 va firing giam that -> co the vuot. LB la trong tai.')
